# Train model RVC giọng của bạn (Colab GPU)

Chạy lần lượt các cell. Yêu cầu: **Runtime → Change runtime type → T4 GPU**.

Chuẩn bị trước: chạy `scripts/prepare_dataset.py` ở máy local để cắt bản thu thành các đoạn 4-10 giây, rồi zip thư mục `dataset/` lại (khuyến nghị 10-30 phút audio sạch, không nhạc nền, không reverb, cùng một mic).

In [ ]:
!nvidia-smi

## 1. Cài RVC WebUI

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc
%cd /content/rvc
!pip install -q -r requirements.txt
!pip install -q gradio==4.44.0 fairseq==0.12.2

## 2. Tải pretrained + hubert + rmvpe

In [ ]:
BASE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main"
!wget -q -P assets/hubert {BASE}/hubert_base.pt
!wget -q -P assets/rmvpe  {BASE}/rmvpe.pt
for f in ["f0G40k.pth", "f0D40k.pth"]:
    !wget -q -P assets/pretrained_v2 {BASE}/pretrained_v2/{f}
!ls -la assets/hubert assets/rmvpe assets/pretrained_v2

## 3. Upload dataset.zip (thư mục dataset đã cắt sẵn)

In [ ]:
from google.colab import files

uploaded = files.upload()  # chọn dataset.zip
!mkdir -p /content/dataset && unzip -q -o -j $(ls *.zip | head -1) -d /content/dataset
!ls /content/dataset | head
!ls /content/dataset | wc -l

## 4. Train

Mở link public gradio in ra bên dưới, vào tab **Train**:

| Trường | Giá trị |
|---|---|
| Experiment name | `my-voice` |
| Target sample rate | `40k` |
| Model có pitch guidance | `true` (bắt buộc cho hát) |
| Version | `v2` |
| Trainset folder | `/content/dataset` |
| f0 method | `rmvpe_gpu` |
| Batch size | `8` (T4) |
| Total epochs | `150-250` |
| Save frequency | `25` |
| Cache dataset to GPU | `No` nếu dataset > 15 phút |

Bấm lần lượt: **Process data** → **Feature extraction** → **Train model** → **Train feature index**.

Mẹo: dừng khi loss đi ngang; train quá lâu sẽ overfit (giọng nghe méo, mất tự nhiên).

In [ ]:
%cd /content/rvc
!python infer-web.py --colab --pycmd python --share

## 5. Tải model về máy

Bỏ 2 file này vào thư mục `models/` của repo `rvc-cover-vi`.

In [ ]:
import glob

from google.colab import files

NAME = "my-voice"
pth = f"/content/rvc/assets/weights/{NAME}.pth"
index = sorted(glob.glob(f"/content/rvc/logs/{NAME}/added_*.index"))
print(pth, index)
files.download(pth)
if index:
    files.download(index[0])